In [1]:
import numpy as np
import pandas as pd

In [2]:
# Load cleaned dataset
input_path = "../data/processed/airline_fraud_cleaned.parquet"

df = pd.read_parquet(input_path, engine="pyarrow")

df.shape

(3000000, 16)

In [3]:
df.columns

Index(['transaction_date', 'route', 'amount', 'amount_in_usd', 'currency',
       'card_type', 'card_bin', 'bank_name', 'bin_country', 'loyalty_tier',
       'txn_origin_country', 'account_age_days', 'failed_attempts', 'is_fraud',
       'billing_country', 'session_time_seconds'],
      dtype='str')

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000000 entries, 0 to 2999999
Data columns (total 16 columns):
 #   Column                Dtype         
---  ------                -----         
 0   transaction_date      datetime64[us]
 1   route                 category      
 2   amount                float64       
 3   amount_in_usd         float64       
 4   currency              category      
 5   card_type             category      
 6   card_bin              category      
 7   bank_name             category      
 8   bin_country           category      
 9   loyalty_tier          category      
 10  txn_origin_country    category      
 11  account_age_days      int16         
 12  failed_attempts       int8          
 13  is_fraud              int8          
 14  billing_country       category      
 15  session_time_seconds  int16         
dtypes: category(9), datetime64[us](1), float64(2), int16(2), int8(2)
memory usage: 114.5 MB


In [5]:
(df.memory_usage(deep=True) / (1024 ** 2)).sort_values(ascending=False)

transaction_date        22.888184
amount_in_usd           22.888184
amount                  22.888184
route                    5.744333
session_time_seconds     5.722046
account_age_days         5.722046
card_bin                 2.861343
txn_origin_country       2.861204
bank_name                2.861132
currency                 2.861096
loyalty_tier             2.861074
card_type                2.861063
bin_country              2.861061
billing_country          2.861061
is_fraud                 2.861023
failed_attempts          2.861023
Index                    0.000126
dtype: float64

## Inspect Daily Statistics

In [6]:
daily_stats = (
    df.groupby(df["transaction_date"].dt.date, observed=True)["is_fraud"]
      .agg(transaction_count="size", fraud_count="sum")
)

daily_stats["fraud_rate"] = daily_stats["fraud_count"] / daily_stats["transaction_count"] * 100
daily_stats

,transaction_count,fraud_count,fraud_rate
transaction_date,,,
2025-12-01,16872,266,1.576577
2025-12-02,16710,215,1.286655
2025-12-03,16543,209,1.263374
2025-12-04,16917,405,2.394041
2025-12-05,16564,265,1.599855
...,...,...,...
2026-05-25,16414,130,0.792007
2026-05-26,16567,313,1.889298
2026-05-27,16611,251,1.511047


### Why did we inspect daily statistics?

Because we're about to make the one and only split of the project.

We already know the split must be chronological, but before locking the boundaries, we want to make sure we're not accidentally creating something like:
- a tiny test period
- a period with unusually low fraud volume
- an artificial temporal break

## Quantify the Candidate Chronological Split

In [7]:
# Candidate chronological split boundaries
n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_end_date = df["transaction_date"].iloc[train_end - 1]
val_end_date = df["transaction_date"].iloc[val_end - 1]

train_end_date, val_end_date

(Timestamp('2026-04-05 23:59:25'), Timestamp('2026-05-03 00:28:02'))

- **Train Set:** From index 0 to 70% ⇰ 70% of the data
- **Validation Set:** From index 70% to 85% ⇰ 15% of the data
- **Test Set:** From index 85% to the end (100%) ⇰ 15% of the data

Our code finds the exact calendar dates where our training and validation sets end.

## Verify Date-Based Split

In [8]:
train = df[df["transaction_date"] < "2026-04-06"]
validation = df[
    (df["transaction_date"] >= "2026-04-06") &
    (df["transaction_date"] < "2026-05-03")
]
test = df[df["transaction_date"] >= "2026-05-03"]

split_stats = pd.DataFrame({
    "rows": [len(train), len(validation), len(test)],
    "fraud_count": [train["is_fraud"].sum(), validation["is_fraud"].sum(), test["is_fraud"].sum()],
    "fraud_rate_%": [
        train["is_fraud"].mean() * 100,
        validation["is_fraud"].mean() * 100,
        test["is_fraud"].mean() * 100
    ]
}, index=["train", "validation", "test"])

split_stats

,rows,fraud_count,fraud_rate_%
train,2100009,31589,1.504232
validation,449656,6653,1.479575
test,450335,6758,1.500661


Those differences are negligible.

More importantly, we gained something much more valuable: Every calendar day belongs entirely to one partition.

That's cleaner than cutting through `2026-04-05 23:59:25` and `2026-05-03 00:28:02`.

## Lock the Split Boundaries

In [9]:
train_end = "2026-04-06"
test_start = "2026-05-03"

train = df[df["transaction_date"] < train_end].copy()
validation = df[
    (df["transaction_date"] >= train_end) &
    (df["transaction_date"] < test_start)
].copy()
test = df[df["transaction_date"] >= test_start].copy()

split_stats

,rows,fraud_count,fraud_rate_%
train,2100009,31589,1.504232
validation,449656,6653,1.479575
test,450335,6758,1.500661


### Why Create Copies before Locking?

Mathematically, they filter the exact same rows. However, computationally and architecturally, they are completely different. The differences come down to Memory Management and Code Maintainability.

1. Memory Management: Views vs. Copies

    * The First Block (No `.copy()`) creates Views.
    * These variables (`train`, `validation`, `test`) are just pointers looking at slices of the original `df`.
        * If we try to modify train later (e.g., handling missing data or encoding variables), pandas will trigger a `SettingWithCopyWarning` and might accidentally overwrite our original `df`.
    * The Second Block (With `.copy()`) creates Deep Copies.
    * This forces pandas to duplicate the data into three completely independent blocks of memory.
        * We can safely modify train without any warnings, and our original `df` remains perfectly untouched.

2. Code Maintainability: Hardcoded vs. Parametrized

    * In our first block we hardcoded Strings.
    * We typed out the date `2026-04-06` multiple times.
        * If we decide to change our training window to `2026-04-01`, we have to manually find and replace that string in multiple places. If we miss one, our validation set logic breaks.
    * Our second block uses Variables.
    * We defined the dates exactly once at the top (`train_end = "2026-04-06"`).
        * If we want to change our time windows later, we only change it in one single place. The rest of our filtering logic updates automatically without risk of human error.

In [10]:
train.shape

(2100009, 16)

In [11]:
train.head()

,transaction_date,route,amount,amount_in_usd,currency,card_type,card_bin,bank_name,bin_country,loyalty_tier,txn_origin_country,account_age_days,failed_attempts,is_fraud,billing_country,session_time_seconds
0,2025-12-01 00:00:04,SIN-ATL,771.18,771.18,USD,Amex,834707,Wells Fargo,US,Silver,US,110,0,0,US,1022
1,2025-12-01 00:00:21,BCN-FRA,250.36,250.36,USD,Amex,514903,HDFC,IN,Gold,IN,582,0,0,IN,903
2,2025-12-01 00:00:21,CJU-SUB,399.36,2.80,JPY,MasterCard,632207,HSBC,GB,Silver,ZA,272,2,0,GB,170
3,2025-12-01 00:00:35,CTS-GMP,137.42,89.32,AUD,Visa,762572,Citi,US,Silver,US,291,0,0,US,511
4,2025-12-01 00:00:37,HAN-SYD,1106.53,1272.51,EUR,Visa,762572,Citi,US,Gold,US,359,0,0,US,912


In [12]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 2100009 entries, 0 to 2100008
Data columns (total 16 columns):
 #   Column                Dtype         
---  ------                -----         
 0   transaction_date      datetime64[us]
 1   route                 category      
 2   amount                float64       
 3   amount_in_usd         float64       
 4   currency              category      
 5   card_type             category      
 6   card_bin              category      
 7   bank_name             category      
 8   bin_country           category      
 9   loyalty_tier          category      
 10  txn_origin_country    category      
 11  account_age_days      int16         
 12  failed_attempts       int8          
 13  is_fraud              int8          
 14  billing_country       category      
 15  session_time_seconds  int16         
dtypes: category(9), datetime64[us](1), float64(2), int16(2), int8(2)
memory usage: 80.1 MB


In [13]:
(train.memory_usage(deep=True) / (1024**2)).sort_values(ascending=False)

transaction_date        16.021797
amount_in_usd           16.021797
amount                  16.021797
route                    4.027737
session_time_seconds     4.005449
account_age_days         4.005449
card_bin                 2.003045
txn_origin_country       2.002906
bank_name                2.002833
currency                 2.002798
loyalty_tier             2.002776
card_type                2.002765
bin_country              2.002763
billing_country          2.002763
is_fraud                 2.002725
failed_attempts          2.002725
Index                    0.000126
dtype: float64

# **Feature Engineering**

## Create Candidate Temporal Features

- Hour of transaction ⇰ unusual transaction timing
- Day of week ⇰ weekday/weekend behavior
- Day of month ⇰ possible billing/pay-cycle effects
- Month ⇰ temporal drift/seasonality

We'll create these in all three partitions using the same deterministic transformation. No information from another row is involved, so there's no leakage here.

In [14]:
for data in (train, validation, test):
    data["transaction_hour"] = data["transaction_date"].dt.hour
    data["transaction_dayofweek"] = data["transaction_date"].dt.dayofweek
    data["transaction_day"] = data["transaction_date"].dt.day
    data["transaction_month"] = data["transaction_date"].dt.month

In [15]:
train[[
    "transaction_date",
    "transaction_hour",
    "transaction_dayofweek",
    "transaction_day",
    "transaction_month"
]].head()

,transaction_date,transaction_hour,transaction_dayofweek,transaction_day,transaction_month
0,2025-12-01 00:00:04,0,0,1,12
1,2025-12-01 00:00:21,0,0,1,12
2,2025-12-01 00:00:21,0,0,1,12
3,2025-12-01 00:00:35,0,0,1,12
4,2025-12-01 00:00:37,0,0,1,12


In [16]:
train[[
    "transaction_date",
    "transaction_hour",
    "transaction_dayofweek",
    "transaction_day",
    "transaction_month"
]].tail()

,transaction_date,transaction_hour,transaction_dayofweek,transaction_day,transaction_month
2100004,2026-04-05 23:59:45,23,6,5,4
2100005,2026-04-05 23:59:48,23,6,5,4
2100006,2026-04-05 23:59:52,23,6,5,4
2100007,2026-04-05 23:59:56,23,6,5,4
2100008,2026-04-05 23:59:56,23,6,5,4


## Evaluate Temporal Feature Signal

In [17]:
temporal_features = [
    "transaction_hour",
    "transaction_dayofweek",
    "transaction_day",
    "transaction_month"
]

for feature in temporal_features:
    print(
        train.groupby(feature, observed=True)["is_fraud"]
        .agg(["size", "sum", "mean"])
        .assign(fraud_rate_pct=lambda x: x["mean"] * 100)
    )

                   size   sum      mean  fraud_rate_pct
transaction_hour                                       
0                 87565  1215  0.013875        1.387541
1                 87500  1513  0.017291        1.729143
2                 87432  1110  0.012696        1.269558
3                 87527  1445  0.016509        1.650919
4                 87807  1464  0.016673        1.667293
5                 87809  1328  0.015124        1.512373
6                 87853  1677  0.019089        1.908870
7                 86787  1173  0.013516        1.351585
8                 88196  1340  0.015193        1.519343
9                 87763  1383  0.015758        1.575835
10                87406  1061  0.012139        1.213875
11                86800  1140  0.013134        1.313364
12                87314  1318  0.015095        1.509494
13                87507  1314  0.015016        1.501594
14                87404  1109  0.012688        1.268821
15                87534  1228  0.014029        1

- `transaction_hour`
    * Lowest: ~1.21%
    * Highest: ~1.91%
    * Overall: ~1.50%

That's not enormous, but it's not completely flat either. Worth keeping as a candidate.

- `transaction_dayofweek`
    * Saturday: ~1.25%
    * Friday: ~1.75%

Again, not a dominant signal, but plausible enough to retain as a candidate.

- `transaction_day`
    * Day 11 → ~1.03%
    * Day 16 → ~2.18%
    * Day 30 → ~1.95%

There's no obvious business/temporal pattern. It's very likely mostly noise. 

**Drop `transaction_day`.**

- `transaction_month`
    * December: full
    * January: full
    * February: full
    * March: full
    * April: only April 1–5

So the apparent monthly differences aren't a useful basis for feature engineering. Also, we're only dealing with ~5 months, so there's very little basis for learning seasonality.

**Drop `transaction_month`.**

But I don't want to call hour/day-of-week final features yet. Let's check whether their signal is reasonably stable in the validation period. If a pattern exists only in training, that's exactly the kind of feature we don't want.

## Check Temporal Signal Stability

In [18]:
for feature in ["transaction_hour", "transaction_dayofweek"]:
    print(
        validation.groupby(feature, observed=True)["is_fraud"]
        .agg(["size", "sum", "mean"])
        .assign(fraud_rate_pct=lambda x: x["mean"] * 100)
    )

                   size  sum      mean  fraud_rate_pct
transaction_hour                                      
0                 18893  380  0.020113        2.011327
1                 18989  420  0.022118        2.211807
2                 18685  186  0.009955        0.995451
3                 18812  277  0.014725        1.472464
4                 18799  278  0.014788        1.478802
5                 18593  244  0.013123        1.312322
6                 18460  261  0.014139        1.413868
7                 18672  464  0.024850        2.485004
8                 18692  376  0.020116        2.011556
9                 18672  227  0.012157        1.215724
10                18653  181  0.009704        0.970353
11                18678  198  0.010601        1.060071
12                18628  340  0.018252        1.825209
13                18956  390  0.020574        2.057396
14                18799  353  0.018778        1.877759
15                18798  314  0.016704        1.670390
16        

- `transaction_hour`: The exact hourly pattern changes between train and validation, but that's expected with a relatively small fraud rate. More importantly, the feature still shows substantial variation rather than collapsing into a flat ~1.5%.
- `transaction_dayofweek`: This is actually reasonably interpretable and remains non-flat in validation. Friday/Saturday behavior changes, but the feature retains variation.

## Remove Unnecessary Temporal Features

In [19]:
for data in (train, validation, test):
    data.drop(columns=["transaction_day", "transaction_month"], inplace=True)

## Inspect Monetary Features

In [20]:
train[["amount", "amount_in_usd"]].describe().T

,count,mean,std,min,25%,50%,75%,max
amount,2100009.0,942.216785,752.293673,1.00,418.24,771.12,1258.59,20869.22
amount_in_usd,2100009.0,816.614532,808.154673,0.01,225.71,634.94,1163.06,23377.71


## Evaluate Monetary Redundancy

In [21]:
train["usd_per_unit"] = train["amount_in_usd"] / train["amount"]

train.groupby("currency", observed=True)["usd_per_unit"].agg(
    ["count", "mean", "std", "min", "max"]
).sort_values("count", ascending=False)

,count,mean,std,min,max
currency,,,,,
USD,841490,1.000,0.000000,1.000000,1.000000
EUR,419077,1.150,0.000038,1.146552,1.153846
GBP,210051,1.300,0.000037,1.296552,1.304762
JPY,210012,0.007,0.000038,0.004673,0.010000
CAD,209647,0.730,0.000041,0.725490,0.733333
AUD,105232,0.650,0.000039,0.647059,0.654545
INR,104500,0.012,0.000046,0.008197,0.015873


## Verify Country Redundancy

In [22]:
for name, data in {
    "train": train,
    "validation": validation,
    "test": test
}.items():
    print(name, data["bin_country"].equals(data["billing_country"]))

train True
validation True
test True


## Remove Redundant Country Feature

In [23]:
for data in (train, validation, test):
    data.drop(columns="bin_country", inplace=True)

In [24]:
train.columns

Index(['transaction_date', 'route', 'amount', 'amount_in_usd', 'currency',
       'card_type', 'card_bin', 'bank_name', 'loyalty_tier',
       'txn_origin_country', 'account_age_days', 'failed_attempts', 'is_fraud',
       'billing_country', 'session_time_seconds', 'transaction_hour',
       'transaction_dayofweek', 'usd_per_unit'],
      dtype='str')

## Evaluate Existing Categorical Features

In [25]:
categorical_features = ["currency", "card_type", "card_bin", "bank_name", "loyalty_tier", "billing_country"]

for feature in categorical_features:
    print(
        train.groupby(feature, observed=True)["is_fraud"]
        .agg(["size", "sum", "mean"])
        .assign(fraud_rate_pct=lambda x: x["mean"] * 100)
        .sort_values("size", ascending=False)
    )

            size    sum      mean  fraud_rate_pct
currency                                         
USD       841490  12662  0.015047        1.504712
EUR       419077   6137  0.014644        1.464409
GBP       210051   3170  0.015092        1.509157
JPY       210012   3145  0.014975        1.497533
CAD       209647   3231  0.015412        1.541162
AUD       105232   1637  0.015556        1.555610
INR       104500   1607  0.015378        1.537799
               size    sum      mean  fraud_rate_pct
card_type                                           
Visa        1050073  15862  0.015106        1.510562
MasterCard   839601  12632  0.015045        1.504524
Amex         210335   3095  0.014715        1.471462
            size   sum      mean  fraud_rate_pct
card_bin                                        
315234    209665  2652  0.012649        1.264875
762572    209303  2581  0.012331        1.233140
552739    209295  2602  0.012432        1.243221
334744    168372  2447  0.014533        

| Feature           | Decision           | Why                                                                        |
| ----------------- | ------------------ | -------------------------------------------------------------------------- |
| `currency`        | **KEEP**           | Weak marginal signal, but important because `amount` is in native currency |
| `card_type`       | **DROP**           | Essentially flat: ~1.47–1.51%                                              |
| `card_bin`        | **KEEP**           | Clear variation: ~1.15–3.75%                                               |
| `bank_name`       | **DROP**           | Essentially flat: ~1.44–1.55%                                              |
| `loyalty_tier`    | **KEEP**           | Consistent fraud-rate gradient                                             |
| `billing_country` | **KEEP candidate** | Weak marginal signal, but meaningful contextual/business feature           |


In [26]:
for data in (train, validation, test):
    data.drop(columns=["card_type", "bank_name"], inplace=True)

## Remove the Synthetic Target Artifact

In [27]:
for data in (train, validation, test):
    data.drop(columns="txn_origin_country", inplace=True)

In [28]:
train.columns

Index(['transaction_date', 'route', 'amount', 'amount_in_usd', 'currency',
       'card_bin', 'loyalty_tier', 'account_age_days', 'failed_attempts',
       'is_fraud', 'billing_country', 'session_time_seconds',
       'transaction_hour', 'transaction_dayofweek', 'usd_per_unit'],
      dtype='str')

## Evaluate Remaining Numerical Features

In [29]:
numeric_features = [
    "account_age_days",
    "failed_attempts",
    "session_time_seconds"
]

train[numeric_features].describe().T

,count,mean,std,min,25%,50%,75%,max
account_age_days,2100009.0,359.639705,254.428710,0.0,173.0,302.0,484.0,3712.0
failed_attempts,2100009.0,0.096054,0.482210,0.0,0.0,0.0,0.0,5.0
session_time_seconds,2100009.0,622.003203,333.500991,45.0,333.0,622.0,911.0,1199.0


## Remove Weak Numerical Features

In [30]:
for data in (train, validation, test):
    data.drop(columns=["failed_attempts", "session_time_seconds"], inplace=True)

In [31]:
train.shape

(2100009, 13)

In [32]:
train.columns

Index(['transaction_date', 'route', 'amount', 'amount_in_usd', 'currency',
       'card_bin', 'loyalty_tier', 'account_age_days', 'is_fraud',
       'billing_country', 'transaction_hour', 'transaction_dayofweek',
       'usd_per_unit'],
      dtype='str')

## Route: First Understand the Frequency Structure

The important question is:

> Are the 1,558 routes sufficiently represented to make a categorical representation sensible?

In [33]:
route_stats = (
    train.groupby("route", observed=True)["is_fraud"]
    .agg(transaction_count="size", fraud_count="sum")
)

route_stats["fraud_rate_pct"] = (
    route_stats["fraud_count"] / route_stats["transaction_count"] * 100
)

route_stats.describe().T

,count,mean,std,min,25%,50%,75%,max
transaction_count,1558.0,1347.887677,1977.785246,1.0,305.000000,752.500000,1574.750000,22918.000000
fraud_count,1558.0,20.275353,29.831370,0.0,4.000000,11.000000,24.000000,331.000000
fraud_rate_pct,1558.0,1.513898,0.828903,0.0,1.182894,1.476385,1.754386,11.111111


## Evaluate Route Frequency

In [34]:
route_stats["transaction_count"].describe(
    percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

count     1558.000000
mean      1347.887677
std       1977.785246
min          1.000000
1%          10.570000
5%          54.000000
10%        111.700000
25%        305.000000
50%        752.500000
75%       1574.750000
90%       3063.300000
95%       4577.650000
99%       9773.510000
max      22918.000000
Name: transaction_count, dtype: float64

For `route`, I don't want to jump straight into target encoding. That would be exactly the kind of flashy feature that can quietly leak.

A much safer candidate is `route_txn_count_before` means how many earlier transactions have occurred on this route before the current transaction.

This uses no fraud labels, so it's inherently safer. And because our data is chronological, we can construct it causally. We should first create it as a candidate, then see whether it actually relates to fraud.

## Create Leakage-Safe Route Activity Feature

In [35]:
all_data = pd.concat(
    [train, validation, test],
    axis=0,
    ignore_index=False
)

all_data["route_txn_count_before"] = (
    all_data.groupby("route", observed=True).cumcount()
)

train = all_data.loc[train.index].copy()
validation = all_data.loc[validation.index].copy()
test = all_data.loc[test.index].copy()

In [36]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 2100009 entries, 0 to 2100008
Data columns (total 14 columns):
 #   Column                  Dtype         
---  ------                  -----         
 0   transaction_date        datetime64[us]
 1   route                   category      
 2   amount                  float64       
 3   amount_in_usd           float64       
 4   currency                category      
 5   card_bin                category      
 6   loyalty_tier            category      
 7   account_age_days        int16         
 8   is_fraud                int8          
 9   billing_country         category      
 10  transaction_hour        int32         
 11  transaction_dayofweek   int32         
 12  usd_per_unit            float64       
 13  route_txn_count_before  int64         
dtypes: category(5), datetime64[us](1), float64(3), int16(1), int32(2), int64(1), int8(1)
memory usage: 114.2 MB


In [37]:
train.route_txn_count_before.tail()

2100004     4629
2100005      478
2100006     7738
2100007    16927
2100008     2316
Name: route_txn_count_before, dtype: int64

## Evaluate Route Activity Signal

In [38]:
route_activity_bins = [-1, 0, 10, 50, 100, 500, 1000, 5000, np.inf]
route_activity_labels = [
    "0",
    "1-10",
    "11-50",
    "51-100",
    "101-500",
    "501-1000",
    "1001-5000",
    "5001+"
]

route_activity_stats = (
    train.assign(
        route_activity_group=pd.cut(
            train["route_txn_count_before"],
            bins=route_activity_bins,
            labels=route_activity_labels
        )
    )
    .groupby("route_activity_group", observed=True)["is_fraud"]
    .agg(["size", "sum", "mean"])
)

route_activity_stats["fraud_rate_pct"] = (
    route_activity_stats["mean"] * 100
)

route_activity_stats

,size,sum,mean,fraud_rate_pct
route_activity_group,,,,
0,1558,16,0.010270,1.026958
1-10,15507,252,0.016251,1.625073
11-50,60500,944,0.015603,1.560331
51-100,72393,1200,0.016576,1.657619
101-500,467670,7222,0.015443,1.544251
501-1000,386052,5828,0.015096,1.509641
1001-5000,857068,12731,0.014854,1.485413
5001+,239261,3396,0.014194,1.419370


### Check Route Activity Stability

In [39]:
validation_route_stats = (
    validation.assign(
        route_activity_group=pd.cut(
            validation["route_txn_count_before"],
            bins=route_activity_bins,
            labels=route_activity_labels
        )
    )
    .groupby("route_activity_group", observed=True)["is_fraud"]
    .agg(["size", "sum", "mean"])
)

validation_route_stats["fraud_rate_pct"] = (
    validation_route_stats["mean"] * 100
)

validation_route_stats

,size,sum,mean,fraud_rate_pct
route_activity_group,,,,
1-10,16,0,0.000000,0.000000
11-50,304,4,0.013158,1.315789
51-100,963,24,0.024922,2.492212
101-500,25249,356,0.014100,1.409957
501-1000,46999,737,0.015681,1.568118
1001-5000,239590,3483,0.014537,1.453733
5001+,136535,2049,0.015007,1.500714


### Remove the Failed Route Activity Candidate

In [40]:
for data in (train, validation, test):
    data.drop(columns="route_txn_count_before", inplace=True)

In [41]:
train.columns

Index(['transaction_date', 'route', 'amount', 'amount_in_usd', 'currency',
       'card_bin', 'loyalty_tier', 'account_age_days', 'is_fraud',
       'billing_country', 'transaction_hour', 'transaction_dayofweek',
       'usd_per_unit'],
      dtype='str')

In [42]:
validation.columns

Index(['transaction_date', 'route', 'amount', 'amount_in_usd', 'currency',
       'card_bin', 'loyalty_tier', 'account_age_days', 'is_fraud',
       'billing_country', 'transaction_hour', 'transaction_dayofweek',
       'usd_per_unit'],
      dtype='str')

In [43]:
for data in (train, validation, test):
    data.drop(columns="amount_in_usd", errors="ignore", inplace=True)

In [44]:
train.columns

Index(['transaction_date', 'route', 'amount', 'currency', 'card_bin',
       'loyalty_tier', 'account_age_days', 'is_fraud', 'billing_country',
       'transaction_hour', 'transaction_dayofweek', 'usd_per_unit'],
      dtype='str')

## Verify the Three-Partition Schema

In [45]:
for name, data in {
    "train": train,
    "validation": validation,
    "test": test
}.items():
    print(name, data.columns.tolist())

train ['transaction_date', 'route', 'amount', 'currency', 'card_bin', 'loyalty_tier', 'account_age_days', 'is_fraud', 'billing_country', 'transaction_hour', 'transaction_dayofweek', 'usd_per_unit']
validation ['transaction_date', 'route', 'amount', 'currency', 'card_bin', 'loyalty_tier', 'account_age_days', 'is_fraud', 'billing_country', 'transaction_hour', 'transaction_dayofweek', 'usd_per_unit']
test ['transaction_date', 'route', 'amount', 'currency', 'card_bin', 'loyalty_tier', 'account_age_days', 'is_fraud', 'billing_country', 'transaction_hour', 'transaction_dayofweek', 'usd_per_unit']


## Evaluate Route Predictive Strength by Frequency

In [46]:
route_fraud_stats = (
    train.groupby("route", observed=True)["is_fraud"]
    .agg(["size", "mean"])
)

route_fraud_stats["fraud_rate_pct"] = route_fraud_stats["mean"] * 100

route_fraud_stats.sort_values("fraud_rate_pct", ascending=False).head(10)

,size,mean,fraud_rate_pct
route,,,
TPE-HKG,18,0.111111,11.111111
CDG-OKA,10,0.100000,10.000000
DXB-CGK,12,0.083333,8.333333
DEL-SYD,53,0.075472,7.547170
MAD-BKK,14,0.071429,7.142857
HND-MAD,29,0.068966,6.896552
AMS-KIX,29,0.068966,6.896552
DEN-DXB,56,0.053571,5.357143
JFK-LAS,94,0.053191,5.319149


## Choose a Leakage-Safe Route Representation

Given what we've seen, I'd choose frequency encoding as the first route representation to test.

Why?

- 1,558 categories ⇰ raw one-hot is unnecessarily wide.
- Route frequency is available without target information.
- It handles rare/unseen routes naturally.
- It's extremely simple.
- We can calculate the mapping from training data only, then apply it to validation/test.

Let's create the training mapping:

In [47]:
route_frequency = train["route"].value_counts(normalize=True)
route_frequency.head()

route
CGK-MAD    0.010913
BCN-JED    0.008612
MCO-SYD    0.008435
DEN-ATL    0.008061
HND-OKA    0.008061
Name: proportion, dtype: float64

## Apply Leakage-Safe Route Frequency Encoding

In [48]:
for data in (train, validation, test):
    data["route_frequency"] = data["route"].map(route_frequency).fillna(0)

In [49]:
train["route_frequency"].describe()

count    2.100009e+06
mean     2.022883e-03
std      2.132390e-03
min      4.761884e-07
25%      6.066641e-04
50%      1.239995e-03
75%      2.681893e-03
max      1.091329e-02
Name: route_frequency, dtype: float64

In [50]:
train.columns

Index(['transaction_date', 'route', 'amount', 'currency', 'card_bin',
       'loyalty_tier', 'account_age_days', 'is_fraud', 'billing_country',
       'transaction_hour', 'transaction_dayofweek', 'usd_per_unit',
       'route_frequency'],
      dtype='str')

## Check Unseen Routes

In [51]:
for name, data in {
    "validation": validation,
    "test": test
}.items():
    unseen = ~data["route"].isin(route_frequency.index)
    print(f"{name}: {unseen.sum():,} unseen routes ({unseen.mean() * 100:.4f}%)")

validation: 0 unseen routes (0.0000%)
test: 0 unseen routes (0.0000%)


## Define the Categorical Feature Set

In [52]:
categorical_features = [
    "route",
    "currency",
    "card_bin",
    "loyalty_tier",
    "billing_country"
]

numeric_features = [
    "amount",
    "account_age_days",
    "transaction_hour",
    "transaction_dayofweek",
    "route_frequency"
]

target = "is_fraud"

# **Feature Selection**

## Check Candidate Feature Redundancy

Before model-based selection, let's check relationships among our numeric candidates. This is primarily to identify redundant information, not to automatically delete correlated features.

In [ ]:
train[numeric_features].corr()

,amount,account_age_days,transaction_hour,transaction_dayofweek,route_frequency
amount,1.000000,0.007184,0.000635,0.000710,0.059030
account_age_days,0.007184,1.000000,0.000545,0.000518,0.000413
transaction_hour,0.000635,0.000545,1.000000,-0.000190,0.000464
transaction_dayofweek,0.000710,0.000518,-0.000190,1.000000,0.000247
route_frequency,0.059030,0.000413,0.000464,0.000247,1.000000


So no numeric candidate needs to be dropped for linear redundancy.

But there's an important limitation: correlation doesn't tell us whether a feature is useful for predicting `is_fraud`, and it doesn't measure nonlinear relationships.

So now we move to something much more meaningful: univariate predictive strength.

## Measure Univariate Feature Signal

In [54]:
from sklearn.feature_selection import mutual_info_classif

In [55]:
X_numeric = train[numeric_features]
y = train[target]

mi_scores = mutual_info_classif(X_numeric, y, random_state=42)

pd.Series(
    mi_scores,
    index=numeric_features
).sort_values(ascending=False)

transaction_dayofweek    0.061756
amount                   0.023213
transaction_hour         0.019958
route_frequency          0.001850
account_age_days         0.001204
dtype: float64

`mutual_info_classif()` needs to know which features are discrete because the underlying MI estimator treats discrete and continuous variables differently. So technically, `auto` isn't the most explicit representation of our feature semantics.

But I would not use `discrete_mask` here. Because:

1. `transaction_hour` and `transaction_dayofweek` are genuinely discrete.
2. `account_age_days` is stored as integer, but semantically it's a count of days and has thousands of possible values. Treating it as discrete isn't necessarily advantageous for this screening.
3. `auto` is perfectly reasonable for a first-pass MI calculation.
4. Most importantly, we already spent `3 min 18.8 sec` computing this. 😂 I don't want to rerun an expensive estimator just to make the argument explicit when it isn't gonna change our feature decision.

But we shouldn't interpret MI scores as "importance" on an absolute scale. They're useful comparatively here, not as a threshold like `MI > 0.01 = keep`.

The important takeaway is:

* `transaction_dayofweek` has the strongest univariate nonlinear association among these.
* `amount` and `transaction_hour` have meaningful signal.
* `route_frequency` and `account_age_days` have weak univariate MI despite earlier evidence suggesting they contain some signal.

That doesn't mean we're dropping the last two. Tree models can extract useful conditional signal that univariate MI misses.

## Encode Categorical Features for Mutual Information

For the categorical candidates, we need to represent categories numerically and explicitly tell MI they're discrete. We'll create temporary integer codes only for this diagnostic, these are not our final model encodings.

In [ ]:
X_categorical = train[categorical_features].apply(
    lambda col: col.cat.codes
)
y = train[target]

categorical_mi = mutual_info_classif(
    X_categorical,
    y,
    discrete_features=True,
    random_state=42
)

pd.Series(
    categorical_mi,
    index=categorical_features
).sort_values(ascending=False)

card_bin           0.000503
route              0.000375
loyalty_tier       0.000303
billing_country    0.000003
currency           0.000002
dtype: float64

Strongest candidates
- `transaction_dayofweek`
- `amount`
- `transaction_hour`
- `card_bin`
- `route`
- `loyalty_tier`

Weaker candidates
- `account_age_days`
- `route_frequency`

Very weak candidates
- `billing_country`
- `currency`

But we are not dropping the bottom ones yet.

The next step should be the one that actually matters most:

> Can these features contribute when considered together?

## Prepare a Training Sample for Model-Based Selection

In [57]:
selection_features = (numeric_features + categorical_features)

selection_sample = train.sample(
    n=300_000,
    random_state=42
)

X_selection = selection_sample[selection_features]
y_selection = selection_sample[target]

X_selection.shape

(300000, 10)

In [58]:
X_selection.info()

<class 'pandas.DataFrame'>
Index: 300000 entries, 470588 to 945985
Data columns (total 10 columns):
 #   Column                 Non-Null Count   Dtype   
---  ------                 --------------   -----   
 0   amount                 300000 non-null  float64 
 1   account_age_days       300000 non-null  int16   
 2   transaction_hour       300000 non-null  int32   
 3   transaction_dayofweek  300000 non-null  int32   
 4   route_frequency        300000 non-null  float64 
 5   route                  300000 non-null  category
 6   currency               300000 non-null  category
 7   card_bin               300000 non-null  category
 8   loyalty_tier           300000 non-null  category
 9   billing_country        300000 non-null  category
dtypes: category(5), float64(2), int16(1), int32(2)
memory usage: 11.5 MB


## Fit Selection-Only XGBoost

In [59]:
from xgboost import XGBClassifier

In [60]:
selection_model = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(1 - y_selection.mean()) / y_selection.mean(),
    tree_method="hist",
    enable_categorical=True,
    random_state=42,
    n_jobs=-1
)

selection_model.fit(
    X_selection,
    y_selection
)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


### Inspect Model-Based Feature Importance

In [61]:
feature_importance = (
    pd.Series(
        selection_model.feature_importances_,
        index=selection_features
    )
    .sort_values(ascending=False)
)

feature_importance

amount                   0.299229
billing_country          0.281676
route                    0.149673
account_age_days         0.109368
card_bin                 0.106251
transaction_dayofweek    0.016844
transaction_hour         0.013537
currency                 0.012241
route_frequency          0.009757
loyalty_tier             0.001426
dtype: float32

We still shouldn't use this single 300K sample to make the final selection. If `billing_country` really matters, it should continue to matter when we change the sample/model seed. If `loyalty_tier` really contributes almost nothing, it should remain near the bottom.

So instead of inventing another selection method, let's repeat this cheaply with a second independent 300K training sample and compare rankings.

That gives us a much stronger statement:

> Feature importance was stable across independent training samples.

### Test Feature-Importance Stability

In [62]:
selection_sample_2 = train.sample(
    n=300_000,
    random_state=2026
)

X_selection_2 = selection_sample_2[selection_features]
y_selection_2 = selection_sample_2[target]

selection_model_2 = XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(1 - y_selection_2.mean()) / y_selection_2.mean(),
    tree_method="hist",
    enable_categorical=True,
    random_state=42,
    n_jobs=-1
)

selection_model_2.fit(X_selection_2, y_selection_2)

feature_importance_2 = (
    pd.Series(
        selection_model_2.feature_importances_,
        index=selection_features
    )
    .sort_values(ascending=False)
)

feature_importance_2

amount                   0.299958
billing_country          0.222737
route                    0.155852
card_bin                 0.131956
account_age_days         0.095648
transaction_hour         0.026628
route_frequency          0.026304
transaction_dayofweek    0.024399
currency                 0.011712
loyalty_tier             0.004806
dtype: float32

| Feature                 | Sample 1 | Sample 2 |
| ----------------------- | -------- | -------- |
| `amount`                |   0.2992 |   0.3000 |
| `billing_country`       |   0.2817 |   0.2227 |
| `route`                 |   0.1497 |   0.1559 |
| `card_bin`              |   0.1063 |   0.1320 |
| `account_age_days`      |   0.1094 |   0.0956 |
| `transaction_hour`      |   0.0135 |   0.0266 |
| `route_frequency`       |   0.0098 |   0.0263 |
| `transaction_dayofweek` |   0.0168 |   0.0244 |
| `currency`              |   0.0122 |   0.0117 |
| `loyalty_tier`          |   0.0014 |   0.0048 |

The exact importance values move, which is normal for tree models, but the broad structure is remarkably stable:
- Strong: `amount`, `billing_country`, `route`, `card_bin`, `account_age_days`
- Weak: `transaction_hour`, `route_frequency`, `transaction_dayofweek`, `currency`
- Very weak: `loyalty_tier`

And this gives us something much stronger than either MI or one XGBoost run.

`billing_country` is the interesting one. During our EDA, marginal fraud rates were almost flat:
```
US  1.52%
GB  1.47%
ES  1.44%
IN  1.53%
```
Yet XGBoost repeatedly gives it ~22–28% of total importance. That means its predictive value is likely conditional/interacting with other variables, or there's some structure in the synthetic data that isn't visible from simple marginal rates. 

Likewise, `loyalty_tier` is now looking increasingly difficult to justify. It had a fraud-rate gradient in EDA, but its model contribution is consistently tiny.

## Compare Feature Importance Across Runs

In [63]:
importance_comparison = pd.DataFrame({
    "sample_1": feature_importance,
    "sample_2": feature_importance_2
})

importance_comparison["mean_importance"] = (
    importance_comparison.mean(axis=1)
)

importance_comparison["rank_1"] = (
    importance_comparison["sample_1"].rank(ascending=False)
)

importance_comparison["rank_2"] = (
    importance_comparison["sample_2"].rank(ascending=False)
)

importance_comparison.sort_values("mean_importance", ascending=False)

,sample_1,sample_2,mean_importance,rank_1,rank_2
amount,0.299229,0.299958,0.299593,1.0,1.0
billing_country,0.281676,0.222737,0.252206,2.0,2.0
route,0.149673,0.155852,0.152762,3.0,3.0
card_bin,0.106251,0.131956,0.119104,5.0,4.0
account_age_days,0.109368,0.095648,0.102508,4.0,5.0
transaction_dayofweek,0.016844,0.024399,0.020622,6.0,8.0
transaction_hour,0.013537,0.026628,0.020082,7.0,6.0
route_frequency,0.009757,0.026304,0.018030,9.0,7.0
currency,0.012241,0.011712,0.011976,8.0,9.0
loyalty_tier,0.001426,0.004806,0.003116,10.0,10.0


- **Definitely keep:**: `amount`, `billing_country`, `route`, `card_bin`, `account_age_days`
- **Keep for now:** `transaction_hour`, `transaction_dayofweek`, `currency`

Not because they're strong, but because there are legitimate reasons not to eliminate them yet:
* `transaction_hour` / `transaction_dayofweek` represent transaction timing and could interact with other fraud patterns.
* `currency` provides context for interpreting `amount`.

- **Very likely drop:**: `loyalty_tier`, `route_frequency`
    - `loyalty_tier` is rank 10 in both model runs. `route_frequency` is weak in both and its standalone route-activity test wasn't convincing.

But before we make anything irreversible, let's make one final sanity check on those two against the target in the full training set, because we already know the model says they're weak and I want the final decision documented by direct evidence.

### Final Check for Weak Candidates

In [65]:
for feature in ["loyalty_tier", "route_frequency"]:
    if feature == "loyalty_tier":
        stats = (
            train.groupby(feature, observed=True)["is_fraud"]
            .agg(["size", "sum", "mean"])
        )
    else:
        stats = (
            train.assign(
                route_frequency_group=pd.qcut(
                    train[feature],
                    q=5,
                    duplicates="drop"
                )
            )
            .groupby("route_frequency_group", observed=True)["is_fraud"]
            .agg(["size", "sum", "mean"])
        )

    stats["fraud_rate_pct"] = stats["mean"] * 100
    print(stats)

                size    sum      mean  fraud_rate_pct
loyalty_tier                                         
Gold          625744  11080  0.017707        1.770692
None          466118   4662  0.010002        1.000176
Platinum      147323   2646  0.017961        1.796054
Silver        860824  13201  0.015335        1.533531
                                      size   sum      mean  fraud_rate_pct
route_frequency_group                                                     
(-0.0009995240000000001, 0.000502]  421742  6371  0.015106        1.510639
(0.000502, 0.000929]                419058  6220  0.014843        1.484281
(0.000929, 0.00163]                 419839  6315  0.015041        1.504148
(0.00163, 0.00322]                  424434  6483  0.015274        1.527446
(0.00322, 0.0109]                   414936  6200  0.014942        1.494206


## Create a Feature-Selection Holdout

In [66]:
from sklearn.model_selection import train_test_split

In [67]:
selection_train, selection_eval = train_test_split(
    selection_sample,
    test_size=0.2,
    random_state=42,
    stratify=selection_sample[target]
)

X_sel_train = selection_train[selection_features]
y_sel_train = selection_train[target]

X_sel_eval = selection_eval[selection_features]
y_sel_eval = selection_eval[target]

X_sel_train.shape, X_sel_eval.shape

((240000, 10), (60000, 10))

## Prepare the Ablation Feature Sets

In [68]:
feature_sets = {
    "all_features": selection_features,
    "without_loyalty_tier": [
        f for f in selection_features if f != "loyalty_tier"
    ],
    "without_route_frequency": [
        f for f in selection_features if f != "route_frequency"
    ],
    "without_both": [
        f for f in selection_features
        if f not in ["loyalty_tier", "route_frequency"]
    ]
}

{k: len(v) for k, v in feature_sets.items()}

{'all_features': 10,
 'without_loyalty_tier': 9,
 'without_route_frequency': 9,
 'without_both': 8}

## Run Feature Ablation Comparison

Feature Ablation Comparison is a method used to find the exact value of a feature by completely removing it from the dataset and measuring how much the model's performance drops.

In [69]:
from sklearn.metrics import average_precision_score

In [70]:
ablation_results = []

for name, features in feature_sets.items():
    model = XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=(1 - y_sel_train.mean()) / y_sel_train.mean(),
        tree_method="hist",
        enable_categorical=True,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        selection_train[features],
        y_sel_train
    )

    y_pred = model.predict_proba(
        selection_eval[features]
    )[:, 1]

    ablation_results.append({
        "feature_set": name,
        "n_features": len(features),
        "pr_auc": average_precision_score(
            y_sel_eval,
            y_pred
        )
    })

ablation_results = (
    pd.DataFrame(ablation_results)
    .sort_values("pr_auc", ascending=False)
    .reset_index(drop=True)
)

ablation_results

,feature_set,n_features,pr_auc
0,without_loyalty_tier,9,0.414656
1,without_both,8,0.414498
2,without_route_frequency,9,0.409743
3,all_features,10,0.406837


## Lock the Final Feature Set

In [71]:
final_features = [
    "amount",
    "billing_country",
    "route",
    "card_bin",
    "account_age_days",
    "currency",
    "transaction_hour",
    "transaction_dayofweek"
]

len(final_features), final_features

(8,
 ['amount',
  'billing_country',
  'route',
  'card_bin',
  'account_age_days',
  'currency',
  'transaction_hour',
  'transaction_dayofweek'])

## Create Final Model-Ready Dataset

In [72]:
model_columns = final_features + [target]

train_model = train[model_columns].copy()
validation_model = validation[model_columns].copy()
test_model = test[model_columns].copy()

## Final Model-Ready Dataset Check

In [75]:
for name, data in {
    "train": train_model,
    "validation": validation_model,
    "test": test_model
}.items():
    print(f"\n{name}\n")
    data.info()
    print(data[target].value_counts())


train

<class 'pandas.DataFrame'>
RangeIndex: 2100009 entries, 0 to 2100008
Data columns (total 9 columns):
 #   Column                 Dtype   
---  ------                 -----   
 0   amount                 float64 
 1   billing_country        category
 2   route                  category
 3   card_bin               category
 4   account_age_days       int16   
 5   currency               category
 6   transaction_hour       int32   
 7   transaction_dayofweek  int32   
 8   is_fraud               int8    
dtypes: category(4), float64(1), int16(1), int32(2), int8(1)
memory usage: 48.1 MB
is_fraud
0    2068420
1      31589
Name: count, dtype: int64

validation

<class 'pandas.DataFrame'>
RangeIndex: 449656 entries, 2100009 to 2549664
Data columns (total 9 columns):
 #   Column                 Non-Null Count   Dtype   
---  ------                 --------------   -----   
 0   amount                 449656 non-null  float64 
 1   billing_country        449656 non-null  category
 2   

## Save Final Model-Ready Parquet Files

In [76]:
from pathlib import Path

In [82]:
project_root = Path.cwd().parent
output_dir = project_root / "data" / "processed" / "modeling"

output_dir.mkdir(parents=True, exist_ok=True)

train_model.to_parquet(
    output_dir / "train.parquet",
    engine="pyarrow",
    index=False
)

validation_model.to_parquet(
    output_dir / "validation.parquet",
    engine="pyarrow",
    index=False
)

test_model.to_parquet(
    output_dir / "test.parquet",
    engine="pyarrow",
    index=False
)

## Final Verification

In [83]:
for name in ["train", "validation", "test"]:
    path = output_dir / f"{name}.parquet"
    data = pd.read_parquet(path, engine="pyarrow")

    print(f"\n{name}.parquet")
    print(f"Shape: {data.shape}")
    print(f"Columns: {data.columns.tolist()}")
    print(f"Missing values: {data.isna().sum().sum()}")
    print(f"Fraud rate: {data[target].mean() * 100:.4f}%")


train.parquet
Shape: (2100009, 9)
Columns: ['amount', 'billing_country', 'route', 'card_bin', 'account_age_days', 'currency', 'transaction_hour', 'transaction_dayofweek', 'is_fraud']
Missing values: 0
Fraud rate: 1.5042%

validation.parquet
Shape: (449656, 9)
Columns: ['amount', 'billing_country', 'route', 'card_bin', 'account_age_days', 'currency', 'transaction_hour', 'transaction_dayofweek', 'is_fraud']
Missing values: 0
Fraud rate: 1.4796%

test.parquet
Shape: (450335, 9)
Columns: ['amount', 'billing_country', 'route', 'card_bin', 'account_age_days', 'currency', 'transaction_hour', 'transaction_dayofweek', 'is_fraud']
Missing values: 0
Fraud rate: 1.5007%
